# HODGE v10a.18 — Arithmetic Geometry / Prime Provenance

Exact small certificate. No A100 is required, although it runs normally in Colab.
It formalizes which denominator primes can arise from SU(3) Haar projectors, representation-energy splittings, reduced resolvents, and linked-cluster arithmetic.

In [ ]:
# HODGE v10a.18 — SU(3) O(u^4) ARITHMETIC-GEOMETRY / PRIME-PROVENANCE CERTIFICATE
# ======================================================================================
#
# Purpose
# -------
# Turn the observed prime structure of the strong-coupling coefficients into a
# theorem-level arithmetic audit rather than numerology.
#
# Core statement certified here:
#
#   At fixed finite perturbative order, every exact coefficient produced by the
#   Hodge/Haar/Feshbach/linked-cluster construction is rational.  Its denominator
#   prime support can come only from:
#
#     (i) exact SU(3) Haar/Gram inverse projectors,
#    (ii) local irrep-projector energy splittings,
#   (iii) reduced-resolvent gaps E0-E_lambda,
#    (iv) source normalization.
#
#   Translation sums, embedding multiplicities and marked Möbius subtraction are
#   integral operations and therefore introduce no NEW denominator primes.
#
# This script builds an explicit order-four SU(3) superset of those arithmetic
# sources directly from the cubic two-magnetic-step geometry.  It then audits all
# exact coefficients already certified by the project.
#
# IMPORTANT:
# The v10a.16 m4 value is numerical.  Its limit_denominator() fraction is tested
# only as a DISPLAY reconstruction; it is NOT promoted to an exact coefficient.
# ======================================================================================

from fractions import Fraction
from collections import defaultdict, Counter
import itertools
import math
import sympy as sp

N = 3
L = 5
E0 = Fraction(8, 3)

gates = []

def gate(name, ok, detail=""):
    ok = bool(ok)
    gates.append((name, ok, str(detail)))
    print(("[PASS] " if ok else "[FAIL] ") + name +
          (f" :: {detail}" if detail != "" else ""))

def factor_frac(q):
    q = Fraction(q)
    return {
        "num": sp.factorint(abs(q.numerator)),
        "den": sp.factorint(q.denominator),
    }

def den_primes(q):
    return set(sp.factorint(Fraction(q).denominator))

def fmt_factor(n):
    n = int(n)
    if n == 0:
        return "0"
    s = "-" if n < 0 else ""
    fs = sp.factorint(abs(n))
    if not fs:
        return s + "1"
    return s + " * ".join(str(p) if e == 1 else f"{p}^{e}" for p, e in sorted(fs.items()))

print("=" * 118)
print("HODGE v10a.18 — SU(3) O(u^4) ARITHMETIC-GEOMETRY / PRIME-PROVENANCE CERTIFICATE")
print("=" * 118)

# ======================================================================================
# I. EXACT MASS / FOLD / VACUUM LEDGER
# ======================================================================================

print("\n[I] EXACT STRONG-COUPLING LEDGER")

M0 = Fraction(8, 3)
M1 = Fraction(1, 1)
M2 = Fraction(11, 306)
M3 = Fraction(-109151, 249696)

E2_A = Fraction(-5945, 612)
N_A = Fraction(511051, 124848)
J_A = Fraction(-48945521, 25468992)
FOLD_A = Fraction(5315003, 140454)

EV4_1 = Fraction(-39, 1280)
VAC_PAIR_LINKED = Fraction(-327, 83776)
VAC_LINKED_MARK = Fraction(-1474623, 1675520)

for name, q in [
    ("m0", M0), ("m1", M1), ("m2", M2), ("m3", M3),
    ("e2_A", E2_A), ("N_A", N_A), ("J_A", J_A), ("fold_A", FOLD_A),
    ("vac e4 one-face", EV4_1), ("vac pair omega4", VAC_PAIR_LINKED),
    ("vac linked around mark", VAC_LINKED_MARK),
]:
    print(f"  {name:24s} = {q}")
    print(f"      numerator   = {fmt_factor(q.numerator)}")
    print(f"      denominator = {fmt_factor(q.denominator)}")

gate("exact Q1 fold identity closes",
     -E2_A * N_A + J_A == FOLD_A,
     -E2_A * N_A + J_A)

gate("lower-order marked subtraction m2 exact",
     E2_A - 13 * Fraction(-3,4) == M2,
     E2_A - 13 * Fraction(-3,4))

gate("lower-order marked subtraction m3 exact",
     -N_A - 13 * Fraction(-9,32) == M3,
     -N_A - 13 * Fraction(-9,32))

# ======================================================================================
# II. EXACT HAAR / GRAM DENOMINATOR SOURCES
# ======================================================================================

print("\n[II] EXACT SU(3) HAAR / GRAM DENOMINATOR SOURCES")

def pinv(p):
    out = [0] * len(p)
    for i, j in enumerate(p):
        out[j] = i
    return tuple(out)

def pcompose(p, q):
    return tuple(p[q[i]] for i in range(len(p)))

def pcycles(p):
    seen = [False] * len(p)
    c = 0
    for i in range(len(p)):
        if not seen[i]:
            c += 1
            j = i
            while not seen[j]:
                seen[j] = True
                j = p[j]
    return c

haar_sources = {}

# Balanced Weingarten k=1,2,3.
for k in (1,2,3):
    ps = list(itertools.permutations(range(k)))
    G = sp.Matrix([
        [sp.Integer(N) ** pcycles(pcompose(pinv(a), b)) for b in ps]
        for a in ps
    ])
    W = G.inv()
    dens = sorted({int(sp.denom(x)) for x in W})
    primes = sorted(set().union(*(set(sp.factorint(d)) for d in dens)))
    haar_sources[f"balanced k={k}"] = (dens, primes)
    print(f"  balanced k={k}: denominators={dens}; primes={primes}")

# Determinant primitive int UUU = eps eps / 6.
haar_sources["pure determinant k=3"] = ([6], [2,3])
print("  pure determinant (3,0)/(0,3): denominators=[6]; primes=[2, 3]")

# Mixed determinant (4,1) projector from the certified v05c/v9.1 matrix.
C41 = sp.Matrix([
    [sp.Rational(1,32), sp.Rational(1,96),-sp.Rational(1,96), sp.Rational(1,96)],
    [sp.Rational(1,96), sp.Rational(1,32), sp.Rational(1,96),-sp.Rational(1,96)],
    [-sp.Rational(1,96),sp.Rational(1,96), sp.Rational(1,32), sp.Rational(1,96)],
    [sp.Rational(1,96),-sp.Rational(1,96),sp.Rational(1,96), sp.Rational(1,32)],
])
dens41 = sorted({int(sp.denom(x)) for x in C41})
pr41 = sorted(set().union(*(set(sp.factorint(d)) for d in dens41)))
haar_sources["mixed determinant (4,1)"] = (dens41, pr41)
print(f"  mixed determinant (4,1)/(1,4): denominators={dens41}; primes={pr41}")

# Pure six-fundamental rank-five projector, built EXACTLY.
def eps3(t):
    if len(set(t)) < 3:
        return 0
    inv = sum(t[i] > t[j] for i in range(3) for j in range(i+1,3))
    return -1 if inv % 2 else 1

parts = []
for comb in itertools.combinations(range(1,6), 2):
    A = (0,) + comb
    B = tuple(i for i in range(6) if i not in A)
    parts.append((A,B))

# Exact Gram matrix can be obtained without numerical rank decisions.
vals = []
for A, B in parts:
    arr = {}
    for idx in itertools.product(range(3), repeat=6):
        v = eps3(tuple(idx[i] for i in A)) * eps3(tuple(idx[i] for i in B))
        if v:
            arr[idx] = v
    vals.append(arr)

G60 = sp.Matrix([
    [sum(vals[i].get(k,0) * vals[j].get(k,0)
         for k in (set(vals[i]) | set(vals[j])))
     for j in range(len(vals))]
    for i in range(len(vals))
])
_, piv60 = G60.rref()
rank60 = len(piv60)
C60 = G60.extract(piv60, piv60).inv()
dens60 = sorted({int(sp.denom(x)) for x in C60})
pr60 = sorted(set().union(*(set(sp.factorint(d)) for d in dens60)))
haar_sources["pure six"] = (dens60, pr60)
print(f"  pure-six (6,0)/(0,6): rank={rank60}; denominators={dens60}; primes={pr60}")

gate("pure-six invariant rank is exactly five", rank60 == 5, rank60)

HAAR_PRIMES = set()
for _, primes in haar_sources.values():
    HAAR_PRIMES.update(primes)
print("  ==> Haar/Gram denominator prime support =", sorted(HAAR_PRIMES))
gate("order-four Haar prime support is {2,3,5}",
     HAAR_PRIMES == {2,3,5}, sorted(HAAR_PRIMES))

# ======================================================================================
# III. CUBIC TWO-STEP REPRESENTATION / GAP CENSUS
# ======================================================================================

print("\n[III] CUBIC TWO-MAGNETIC-STEP IRREP / ENERGY-GAP CENSUS")

verts = [(x,y,z) for x in range(L) for y in range(L) for z in range(L)]

def shift(v,d,step=1):
    w = list(v)
    w[d] = (w[d] + step) % L
    return tuple(w)

links = []
lid = {}
for v in verts:
    for d in range(3):
        lid[(v,d)] = len(links)
        links.append((v,d))

faces = []
for v in verts:
    for a,b in ((0,1),(0,2),(1,2)):
        faces.append((v,a,b))

def face_flux(f, orient=1):
    v,a,b = faces[f]
    va = shift(v,a)
    vb = shift(v,b)
    seq = [
        (lid[(v,a)], +1),
        (lid[(va,b)], +1),
        (lid[(vb,a)], -1),
        (lid[(v,b)], -1),
    ]
    return {l:int(orient)*s for l,s in seq}

link_faces = [[] for _ in links]
for f in range(len(faces)):
    for l in face_flux(f):
        link_faces[l].append(f)

def candidate_faces(face_set):
    out = set()
    for f in face_set:
        for l in face_flux(f):
            out.update(link_faces[l])
    return out

def e_link(rep):
    p,q = rep
    return Fraction(p*p + q*q + p*q + 3*p + 3*q, 6)

def e_sig(sig):
    return sum((e_link(r) for r in sig.values()), Fraction(0))

def fuse(rep, sg):
    p,q = rep
    out = []
    if sg > 0:
        out.append((p+1,q))
        if p >= 1:
            out.append((p-1,q+1))
        if q >= 1:
            out.append((p,q-1))
    else:
        out.append((p,q+1))
        if q >= 1:
            out.append((p+1,q-1))
        if p >= 1:
            out.append((p-1,q))
    return tuple(out)

# Collect BOTH kinds of denominator source introduced by representation propagation:
# 1. local projector sibling splittings (lambda_i-lambda_j)^(-1)
# 2. reduced resolvents (E0-E_signature)^(-1)
PROJECTOR_GAPS = set()
RESOLVENT_GAPS_1 = set()
RESOLVENT_GAPS_2 = set()

def apply_signature(sig, f, orient):
    fd = face_flux(f, orient)
    chans = [dict(sig)]
    for l, sg in fd.items():
        nxt = []
        for sm in chans:
            r = sm.get(l, (0,0))
            opts = fuse(r, sg)
            eigs = tuple(e_link(x) for x in opts)
            for a,b in itertools.combinations(eigs,2):
                if a != b:
                    PROJECTOR_GAPS.add(a-b)
            for rr in opts:
                zz = dict(sm)
                if rr == (0,0):
                    zz.pop(l, None)
                else:
                    zz[l] = rr
                nxt.append(zz)
        # exact identity de-duplication
        uniq = {}
        for z in nxt:
            key = tuple(sorted((l,p,q) for l,(p,q) in z.items()))
            uniq[key] = z
        chans = list(uniq.values())
    return chans

root = next(i for i,x in enumerate(faces) if x == ((0,0,0),0,1))
sig0 = {l:((1,0) if s>0 else (0,1)) for l,s in face_flux(root,+1).items()}
gate("marked source energy is E0=8/3", e_sig(sig0) == E0, e_sig(sig0))

sig1_records = []
first_faces = candidate_faces({root})
for f1 in first_faces:
    for o1 in (-1,+1):
        for s1 in apply_signature(sig0, f1, o1):
            E = e_sig(s1)
            if E != E0:
                RESOLVENT_GAPS_1.add(E0-E)
            sig1_records.append((f1,o1,s1))

second_energies = set()
for f1,o1,s1 in sig1_records:
    for f2 in candidate_faces({root,f1}):
        for o2 in (-1,+1):
            for s2 in apply_signature(s1, f2, o2):
                E = e_sig(s2)
                second_energies.add(E)
                if E != E0:
                    RESOLVENT_GAPS_2.add(E0-E)

def inverse_den_prime_support(gaps):
    ps = set()
    rows = []
    for g in sorted(gaps):
        inv = Fraction(1,1) / g
        fs = sp.factorint(inv.denominator)
        ps.update(fs)
        rows.append((g,inv,fs))
    return ps, rows

PJP, pjrows = inverse_den_prime_support(PROJECTOR_GAPS)
R1P, r1rows = inverse_den_prime_support(RESOLVENT_GAPS_1)
R2P, r2rows = inverse_den_prime_support(RESOLVENT_GAPS_2)

print("  first-step candidate faces =", len(first_faces))
print("  first-step representation records =", len(sig1_records))
print("  distinct second-step energies =", len(second_energies))
print("  local projector splitting primes =", sorted(PJP))
print("  R1 reduced-resolvent gap primes =", sorted(R1P))
print("  R2 reduced-resolvent gap primes =", sorted(R2P))

# This census deliberately over-approximates the physical corpus: it keeps every
# irrep branch before Haar/center cancellations.  Therefore its prime set is a
# rigorous SUPERSET for the actual order-four production denominator sources.
ARITH_PRIMES = set(HAAR_PRIMES) | set(PJP) | set(R1P) | set(R2P) | {2}
print("  ==> rigorous O(u^4) denominator-prime SUPERSET =", sorted(ARITH_PRIMES))

# ======================================================================================
# IV. ARITHMETIC-LOCALIZATION THEOREM GATES
# ======================================================================================

print("\n[IV] ARITHMETIC-LOCALIZATION THEOREM GATES")

exact_ledger = {
    "m0": M0,
    "m1": M1,
    "m2": M2,
    "m3": M3,
    "e2_A": E2_A,
    "N_A": N_A,
    "J_A": J_A,
    "fold_A": FOLD_A,
    "vac_e4_one": EV4_1,
    "vac_pair_omega4": VAC_PAIR_LINKED,
    "vac_linked_mark": VAC_LINKED_MARK,
    "t3": Fraction(5,612),
    "R2_hop": Fraction(1975,124848),
    "A3": Fraction(5,48),
    "alpha3": Fraction(5,12),
    "b_det": Fraction(-55,13872),
    "det_flat": Fraction(55,3468),
}

for name,q in exact_ledger.items():
    dp = den_primes(q)
    gate(f"{name} denominator primes lie in O4 arithmetic localization",
         dp <= ARITH_PRIMES,
         f"{sorted(dp)} <= {sorted(ARITH_PRIMES)}")

# Integer graph operations cannot make a new denominator prime.
# Demonstrate on the actual marked-vacuum construction:
vac_rebuilt = 13*EV4_1 + 124*VAC_PAIR_LINKED
gate("marked vacuum linked value reconstructed by integer embeddings",
     vac_rebuilt == VAC_LINKED_MARK, vac_rebuilt)
gate("integer embedding/Mobius arithmetic introduces no new denominator primes",
     den_primes(VAC_LINKED_MARK) <= (den_primes(EV4_1) | den_primes(VAC_PAIR_LINKED)),
     f"{sorted(den_primes(VAC_LINKED_MARK))}")

# ======================================================================================
# V. WHAT THE NUMERICAL m4 FRACTION DOES — AND DOES NOT — MEAN
# ======================================================================================

print("\n[V] NUMERICAL m4 DISPLAY-FRACTION FIREWALL")

M4_FLOAT = -11.068479463777946
M4_DISPLAY = Fraction(M4_FLOAT).limit_denominator(10**9)

print("  v10a.16 numerical m4_rest =", repr(M4_FLOAT))
print("  arbitrary display rational =", M4_DISPLAY)
print("    numerator factorization   =", fmt_factor(M4_DISPLAY.numerator))
print("    denominator factorization =", fmt_factor(M4_DISPLAY.denominator))
display_den_primes = den_primes(M4_DISPLAY)
alien = display_den_primes - ARITH_PRIMES
print("  display denominator primes =", sorted(display_den_primes))
print("  primes outside rigorous O4 source superset =", sorted(alien))

# This is the key falsification of "read geometry off the limit_denominator fraction".
gate("arbitrary m4 display fraction contains arithmetic-provenance violation",
     len(alien) > 0,
     f"alien primes={sorted(alien)}")

print("""
INTERPRETATION:
  The numerical m4 value may still be correct.  What fails is the claim that the
  arbitrary Fraction.limit_denominator reconstruction is its exact rational form.

  Any exact O(u^4) coefficient generated by this finite SU(3) Hodge/Haar/Feshbach
  algebra must have denominator primes drawn from the finite provenance set printed
  above.  A reconstructed denominator carrying an alien prime is therefore a
  certificate that the displayed fraction is numerical scaffolding, not physics.
""")

# ======================================================================================
# VI. THEOREM STATEMENT
# ======================================================================================

print("\n[VI] CERTIFIED THEOREM STATEMENT")
print(r"""
ARITHMETIC LOCALIZATION THEOREM — finite-order SU(3) Hodge solver.

Fix a finite connected marked plaquette corpus and a perturbative order r.
Assume:
  (a) Wilson-network fusion is generated by exact SU(3) tensor-product rules;
  (b) Haar physicalization uses exact finite Gram/projector inverses;
  (c) Feshbach reduction uses nonzero exact electric gaps;
  (d) linked subtraction uses integer embedding/Mobius multiplicities.

Then every perturbative coefficient is rational and belongs to the localization

    Z[S_r^{-1}],

where S_r is generated by the denominator primes of the finite Haar/Gram
projectors, the inverse local irrep splittings, the inverse reduced-resolvent
gaps, and source normalization.

In particular, translation sums and linked-cluster combinatorics can change
numerators and prime EXPONENTS in denominators, but cannot introduce a new
denominator prime outside S_r.

For the present SU(3), O(u^4) cubic two-step corpus, this program prints an
explicit rigorous superset S_4.
""")

print("\n" + "=" * 118)
print("FINAL v10a.18 GATE SUMMARY")
print("=" * 118)
for i,(name,ok,detail) in enumerate(gates,1):
    print(f"{i:02d}. {'PASS' if ok else 'FAIL'} — {name}" +
          (f" :: {detail}" if detail else ""))
passed = sum(ok for _,ok,_ in gates)
print("-" * 118)
print(f"PASSED {passed}/{len(gates)} v10a.18 GATES")
if passed != len(gates):
    raise AssertionError("v10a.18 arithmetic-provenance gate failure")

print("""
V10A.18 CONCLUSION
------------------
* Prime structure is now a falsifiable arithmetic statement, not a geometric metaphor.
* Haar geometry contributes only {2,3,5} at this order.
* Electric representation/projector gaps enlarge the allowed denominator-prime set.
* Linked/embedding geometry contributes integer multiplicities and therefore no new denominator primes.
* The arbitrary rational reconstruction of the numerical m4 candidate is explicitly rejected as noncanonical.
* The next exactification target is D_A itself: accumulate the pair-collapsed topology ledger in exact rational arithmetic.
""")
